In [1]:
# Sel ini menghasilkan TIGA dataset cabang (kota) terpisah untuk Tugas Mandiri Pertemuan 3
import numpy as np
import pandas as pd

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")

cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)  # seed berbeda tiap kota agar datanya bervariasi, namun tetap konsisten/reproducible
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat: {df_cabang.shape[0]} baris")

print("\nKetiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.")

Berkas 'transaksi_magelang.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_yogyakarta.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_semarang.csv' berhasil dibuat: 200 baris

Ketiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.


In [2]:
import subprocess

username = subprocess.check_output(
    ["whoami"], text=True
).strip()

print("Username Linux", username)

Username Linux nico


In [3]:
subprocess.run([
    "hdfs", "dfs", "-mkdir", "-p",
    f"/user/{username}/ecommerce/raw"
], check=True)

subprocess.run([
    "hdfs", "dfs", "-mkdir", "-p",
    f"/user/{username}/ecommerce/processed"
], check=True)

print("Struktur direktori berhasil dibuat!")

Struktur direktori berhasil dibuat!


In [5]:
subprocess.run([
    "hdfs", "dfs", "-ls", "-R",
    f"/user/{username}/ecommerce"
])

drwxr-xr-x   - nico supergroup          0 2026-09-09 05:39 /user/nico/ecommerce/processed
drwxr-xr-x   - nico supergroup          0 2026-09-09 05:39 /user/nico/ecommerce/raw


CompletedProcess(args=['hdfs', 'dfs', '-ls', '-R', '/user/nico/ecommerce'], returncode=0)

In [6]:
import os

print("File CSV di folder saat ini:")
print(os.listdir())

File CSV di folder saat ini:
['Modul_Praktikum_BigData_Pertemuan3.ipynb', 'hadoop-3.4.3.tar.gz', 'hasil_unduh_dari_hdfs.txt', 'contoh_lokal.txt', 'transaksi_yogyakarta.csv', 'transaksi_magelang.csv', 'untitled.txt', 'hadoop', 'Tugas3_2505060023_Dimas_Galih_Prakoso.ipynb', 'transaksi_semarang.csv', '.ipynb_checkpoints', 'Modul_Praktikum_BigData_Pertemuan2.ipynb', 'Untitled.ipynb']


In [7]:
for file in [
    "transaksi_magelang.csv",
    "transaksi_yogyakarta.csv",
    "transaksi_semarang.csv"
]:
    subprocess.run([
        "hdfs", "dfs", "-put", "-f",
        file,
        f"/user/{username}/ecommerce/raw/"
    ], check=True)

print("Ketiga CSV berhasil di-upload ke HDFS!")

Ketiga CSV berhasil di-upload ke HDFS!


In [8]:
subprocess.run([
    "hdfs", "dfs", "-ls", "-h",
    f"/user/{username}/ecommerce/raw"
])

Found 3 items
-rw-r--r--   1 nico supergroup     12.0 K 2026-09-09 05:46 /user/nico/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 nico supergroup     11.9 K 2026-09-09 05:46 /user/nico/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 nico supergroup     12.4 K 2026-09-09 05:46 /user/nico/ecommerce/raw/transaksi_yogyakarta.csv


CompletedProcess(args=['hdfs', 'dfs', '-ls', '-h', '/user/nico/ecommerce/raw'], returncode=0)

In [10]:
from io import StringIO

def read_hdfs_csv(path):
    result = subprocess.run(
        ["hdfs", "dfs", "-cat", path],
        capture_output=True,
        text=True,
        check=True
    )

    return pd.read_csv(StringIO(result.stdout))

In [11]:
raw_path = f"/user/{username}/ecommerce/raw"

df_magelang = read_hdfs_csv(
    f"{raw_path}/transaksi_magelang.csv"
)

df_yogyakarta = read_hdfs_csv(
    f"{raw_path}/transaksi_yogyakarta.csv"
)

df_semarang = read_hdfs_csv(
    f"{raw_path}/transaksi_semarang.csv"
)

print("Data Magelang:", df_magelang.shape)
print("Data Yogyakarta:", df_yogyakarta.shape)
print("Data Semarang:", df_semarang.shape)

Data Magelang: (200, 7)
Data Yogyakarta: (200, 7)
Data Semarang: (200, 7)


In [12]:
df_gabungan = pd.concat(
    [
        df_magelang,
        df_yogyakarta,
        df_semarang
    ],
    ignore_index=True
)

print("Jumlah seluruh data:", len(df_gabungan))

display(df_gabungan.head())

Jumlah seluruh data: 600


,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang


In [13]:
print("Jumlah transaksi berdasarkan kota:")

display(
    df_gabungan["kota"].value_counts()
)

Jumlah transaksi berdasarkan kota:


kota
Magelang      200
Yogyakarta    200
Semarang      200
Name: count, dtype: int64

In [14]:
df_gabungan["total_pendapatan"] = (
    df_gabungan["unit_terjual"] *
    df_gabungan["harga_satuan"]
)

display(df_gabungan.head())

,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota,total_pendapatan
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang,50000
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang,175000
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang,175000
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang,150000
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang,700000


In [19]:
ringkasan = (
    df_gabungan
    .groupby(["kota", "kategori"])["total_pendapatan"]
    .sum()
    .reset_index()
        )
display(ringkasan)

,kota,kategori,total_pendapatan
0,Magelang,Elektronik,18775000
1,Magelang,Fashion,27750000
2,Magelang,Kesehatan & Kecantikan,17375000
3,Magelang,Makanan & Minuman,17525000
4,Magelang,Rumah Tangga,12200000
5,Semarang,Elektronik,21425000
6,Semarang,Fashion,26425000
7,Semarang,Kesehatan & Kecantikan,13525000
8,Semarang,Makanan & Minuman,19750000
9,Semarang,Rumah Tangga,10975000


In [20]:
df_gabungan.to_csv(
    "data_gabungan_bersih.csv",
    index=False
)

ringkasan.to_csv(
    "ringkasan_kota_kategori.csv",
    index=False
)

print("Dua file berhasil dibuat:")
print("- data_gabungan_bersih.csv")
print("- ringkasan_kota_kategori.csv")

Dua file berhasil dibuat:
- data_gabungan_bersih.csv
- ringkasan_kota_kategori.csv


In [22]:
processed_path = f"/user/{username}/ecommerce/processed"

for file in [
    "data_gabungan_bersih.csv",
    "ringkasan_kota_kategori.csv"
]:
    subprocess.run([
        "hdfs", "dfs", "-put", "-f",
        file,
        processed_path
    ], check=True)

print("kedua file berhasil di-upload ke HDFS!")

kedua file berhasil di-upload ke HDFS!


In [23]:
subprocess.run([
    "hdfs", "dfs", "-ls", "-h",
    processed_path
])

Found 2 items
-rw-r--r--   1 nico supergroup     40.3 K 2026-09-09 06:20 /user/nico/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 nico supergroup        530 2026-09-09 06:20 /user/nico/ecommerce/processed/ringkasan_kota_kategori.csv


CompletedProcess(args=['hdfs', 'dfs', '-ls', '-h', '/user/nico/ecommerce/processed'], returncode=0)

### Refleksi
Pemisahan data raw dan processed pada HDFS memberikan beberapa keuntungan dalam pengelolaan data. Folder raw digunakan untuk menyimpan data asli yang belum mengalami perubahan, sehingga data tersebut dapat digunakan kembali sebagai sumber apabila di;perlukan proses pengolahan ulang. Sementara itu, folder processed digunakan untuk menyimpan data yang telah dibersihkan, digabungkan, atau diolah sesuai kebutuhan analisis. Pemisahan ini membuat struktur penyimpanan data menjadi lebih teratur dan memudahkan pengguna dalam membedakan data mentah dengan data hasil pengolahan. Selain itu, pemisahan raw dan processed dapat mengurangi risiko data asli tertimpa oleh hasil pengolahan. Dalam proses analisis big data ,struktur seperti ini juga membantu proses pelacakan sumber data pengolahaan data ketika jumlah data semakin banyak.